# Summit.OS — Visual Detector Training

1. Enable GPU: Settings → Accelerator → GPU T4 x2
2. Run All Cells
3. Download outputs from the Output tab when done

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics onnx onnxruntime
print('Done.')

In [ ]:
import os
import json
import urllib.request
import zipfile
import random
from pathlib import Path

BASE   = '/kaggle/working'
SUMMIT = f'{BASE}/summit_detector'
for d in ['images/train','images/val','labels/train','labels/val']:
    os.makedirs(f'{SUMMIT}/{d}', exist_ok=True)

COCO_TO_VISUAL = {1:3, 9:12, 21:17, 22:17, 23:17, 24:17, 3:12, 8:12, 16:17, 17:17, 18:17, 19:17}

print('Downloading COCO annotations...')
urllib.request.urlretrieve(
    'http://images.cocodataset.org/annotations/annotations_trainval2017.zip',
    f'{BASE}/annot.zip'
)
with zipfile.ZipFile(f'{BASE}/annot.zip') as z:
    z.extract('annotations/instances_val2017.json', BASE)
os.remove(f'{BASE}/annot.zip')

with open(f'{BASE}/annotations/instances_val2017.json') as f:
    coco = json.load(f)

img_anns = {}
for ann in coco['annotations']:
    cid = ann['category_id']
    if cid not in COCO_TO_VISUAL: continue
    iid = ann['image_id']
    if iid not in img_anns: img_anns[iid] = []
    img_anns[iid].append(ann)

id2info = {img['id']: img for img in coco['images']}
ids = list(img_anns.keys())
random.seed(42); random.shuffle(ids)
split = int(len(ids) * 0.8)
train_ids, val_ids = ids[:split], ids[split:]

train_fnames = set()
val_fnames   = set()

for split_name, split_ids, fname_set in [('train', train_ids, train_fnames), ('val', val_ids, val_fnames)]:
    for iid in split_ids:
        info = id2info[iid]
        w, h = info['width'], info['height']
        lines = []
        for ann in img_anns[iid]:
            cls = COCO_TO_VISUAL[ann['category_id']]
            x, y, bw, bh = ann['bbox']
            lines.append(f"{cls} {(x+bw/2)/w:.6f} {(y+bh/2)/h:.6f} {bw/w:.6f} {bh/h:.6f}")
        fname = info['file_name']
        with open(f'{SUMMIT}/labels/{split_name}/{fname.replace(".jpg",".txt")}', 'w') as f:
            f.write('\n'.join(lines))
        fname_set.add(fname)

print(f'Labels written — train:{len(train_fnames)}  val:{len(val_fnames)}')
print('Downloading COCO images (~1 GB)...')
urllib.request.urlretrieve(
    'http://images.cocodataset.org/zips/val2017.zip',
    f'{BASE}/images.zip'
)
with zipfile.ZipFile(f'{BASE}/images.zip') as z:
    for fname in train_fnames | val_fnames:
        member = f'val2017/{fname}'
        split_name = 'train' if fname in train_fnames else 'val'
        with z.open(member) as src, open(f'{SUMMIT}/images/{split_name}/{fname}', 'wb') as dst:
            dst.write(src.read())
os.remove(f'{BASE}/images.zip')
print('COCO download complete.')

In [ ]:
import json

yaml_path = f'{SUMMIT}/summit_detector.yaml'
with open(yaml_path, 'w') as f:
    f.write(f"""path: {SUMMIT}
train: images/train
val: images/val
nc: 18
names: [smoke, fire, fire_front, person, person_water, life_raft,
        oil_spill, pipeline_damage, chemical_plume, crop_disease,
        pest_damage, dry_field, vessel, vessel_distress,
        power_line_damage, structural_crack, solar_defect, dangerous_animal]
""")
print('YAML written.')
print(f'Train: {len(list(Path(SUMMIT+"/images/train").iterdir()))} images')
print(f'Val:   {len(list(Path(SUMMIT+"/images/val").iterdir()))} images')

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=32,
    device='0,1',
    project=f'{BASE}/runs',
    name='summit_detector',
    patience=30,
    save=True,
    optimizer='AdamW',
    lr0=1e-3,
    weight_decay=5e-4,
    hsv_h=0.02, hsv_s=0.7, hsv_v=0.4,
    degrees=30, scale=0.6,
    flipud=0.5, fliplr=0.5,
    mosaic=1.0, mixup=0.1,
)
print('Training done.')

In [ ]:
import shutil
import json
import os

best = f'{BASE}/runs/summit_detector/weights/best.pt'
export_model = YOLO(best)
export_model.export(format='onnx', imgsz=640, opset=12, simplify=True)

shutil.copy(best.replace('.pt', '.onnx'), f'{BASE}/summit_detector.onnx')

classes = {
    '0':'smoke','1':'fire','2':'fire_front','3':'person','4':'person_water',
    '5':'life_raft','6':'oil_spill','7':'pipeline_damage','8':'chemical_plume',
    '9':'crop_disease','10':'pest_damage','11':'dry_field','12':'vessel',
    '13':'vessel_distress','14':'power_line_damage','15':'structural_crack',
    '16':'solar_defect','17':'dangerous_animal'
}
with open(f'{BASE}/summit_detector_classes.json', 'w') as f:
    json.dump(classes, f, indent=2)

print(f'ONNX: {os.path.getsize(BASE+"/summit_detector.onnx")/1e6:.1f} MB')
print(f'Config: {BASE}/summit_detector_classes.json')
print()
print('Download from the Output tab (right sidebar).')